# Peer Graded Assignment — Boston Housing Data Analysis

**Statistics for Data Science | Final Project**

This notebook analyzes the historical Boston housing dataset and answers all three assignment tasks. The housing prices are historical, not present-day market valuations. Results describe associations rather than establishing causality.

## Setup: import libraries and load the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
# Course dataset; use a local copy when working offline.
from pathlib import Path
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/boston_housing.csv"
local_csv = Path("boston_housing.csv")
if local_csv.exists():
    boston_df = pd.read_csv(local_csv)
else:
    boston_df = pd.read_csv(url)

# The course dataset may include an extra index column; it is not an explanatory variable.
boston_df = boston_df.loc[:, ~boston_df.columns.str.match(r"^Unnamed")]
print("Dataset shape:", boston_df.shape)
print("Columns:", list(boston_df.columns))
display(boston_df.head())

## Task 1 — Familiarize yourself with the dataset

### Question 1. Display the first five rows

In [ ]:
display(boston_df.head(5))

### Question 2. Display the data types of all columns

In [ ]:
display(boston_df.dtypes.to_frame("data_type"))

### Question 3. Check missing values per column

In [ ]:
display(boston_df.isnull().sum().to_frame("missing_values"))

### Question 4. Check duplicate rows

In [ ]:
print("Number of duplicate records:", boston_df.duplicated().sum())

### Question 5. Display correlation matrix

In [ ]:
correlations = boston_df.corr(numeric_only=True)
display(correlations.round(3))
plt.figure(figsize=(12, 9))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Boston housing: correlation matrix")
plt.tight_layout()
plt.show()

## Task 2 — Descriptive statistics and visualizations

### Question 6. Descriptive statistics

In [ ]:
display(boston_df.describe(include="all").T)

### Question 7. Boxplot of MEDV

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=boston_df["MEDV"], color="skyblue")
plt.title("Distribution of median home values (MEDV)")
plt.xlabel("MEDV (thousands of USD)")
plt.tight_layout()
plt.show()
print("Median MEDV:", round(boston_df["MEDV"].median(), 2))
print("Maximum MEDV:", boston_df["MEDV"].max())

**Interpretation:** The boxplot shows the median, spread, and outlying housing values. MEDV is top-coded at 50 in this historical dataset, so interpretations of the upper tail require care.

### Question 8. Bar plot for CHAS

In [ ]:
chas_counts = boston_df["CHAS"].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=["Not bounded (0)", "Bounded (1)"], y=chas_counts.to_numpy(), ax=ax, color="steelblue")
ax.set_title("Number of census tracts by Charles River boundary")
ax.set_xlabel("CHAS")
ax.set_ylabel("Count of tracts")
plt.tight_layout()
plt.show()
print("Counts by CHAS:")
display(chas_counts.to_frame("count"))

**Interpretation:** The CHAS=0 category is much more common than CHAS=1; the groups are imbalanced, so this should be considered when comparing home values.

### Question 9. Group AGE and create MEDV boxplots

In [ ]:
# AGE = percentage of owner-occupied units built prior to 1940.
# Use explicit percentage bands to make the groups reproducible.
boston_df["AGE_GROUP"] = pd.cut(
    boston_df["AGE"],
    bins=[-np.inf, 35, 70, np.inf],
    labels=["35% or less", "Over 35% to 70%", "Over 70%"],
    include_lowest=True
)
plt.figure(figsize=(9, 5))
sns.boxplot(data=boston_df, x="AGE_GROUP", y="MEDV", color="lightgreen")
plt.title("MEDV by percentage of older owner-occupied units")
plt.xlabel("Percentage of units built before 1940 (AGE group)")
plt.ylabel("MEDV (thousands of USD)")
plt.tight_layout()
plt.show()
display(boston_df.groupby("AGE_GROUP", observed=True)["MEDV"].agg(["size", "median", "mean"]).round(2))

**Interpretation:** Compare the median lines and ranges in the three groups. Tracts with larger shares of older housing generally show lower typical MEDV; the formal ANOVA below tests differences in group **means**.

### Question 10. Scatter plot of NOX versus INDUS

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=boston_df, x="NOX", y="INDUS", alpha=.75)
plt.title("NOX vs. INDUS")
plt.xlabel("Nitric oxide concentration (parts per 10 million)")
plt.ylabel("Non-retail business acres per town (%)")
plt.tight_layout()
plt.show()
print("Pearson correlation (NOX, INDUS):", round(boston_df[["NOX", "INDUS"]].corr().iloc[0, 1], 3))

**Interpretation:** The scatter plot suggests a positive association: towns with higher non-retail business acreage tend to have higher NOX concentrations. Correlation does not demonstrate causation.

### Question 11. Histogram of PTRATIO

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=boston_df, x="PTRATIO", bins=15, color="slateblue", edgecolor="white")
plt.title("Distribution of pupil–teacher ratios")
plt.xlabel("PTRATIO")
plt.ylabel("Number of census tracts")
plt.tight_layout()
plt.show()
print(boston_df["PTRATIO"].describe().round(2))

**Interpretation:** The histogram displays the frequency of school pupil–teacher ratios across census tracts. Peaks indicate commonly observed ratios; it is not a histogram of MEDV.

## Task 3 — Apply statistical tests

**Decision rule:** Statistical significance is assessed at α = 0.05. Statistical tests concern associations or group differences in this observational dataset, not causal effects.

### Question 12. Levene test for equal MEDV variances (CHAS groups)

In [ ]:
chas0 = boston_df.loc[boston_df["CHAS"] == 0, "MEDV"].dropna()
chas1 = boston_df.loc[boston_df["CHAS"] == 1, "MEDV"].dropna()
lev_stat, lev_p = stats.levene(chas0, chas1, center="mean")
print(f"Levene statistic = {lev_stat:.4f}; p-value = {lev_p:.6g}")
print("H0: equal population variances")
print("Decision:", "Reject H0; variances differ." if lev_p < .05 else "Do not reject H0; unequal variances are not established.")

### Question 13. Independent-samples t-test for CHAS vs MEDV

In [ ]:
# Use the variance result to choose Student's t-test or Welch's t-test.
t_stat, t_p = stats.ttest_ind(chas0, chas1, equal_var=(lev_p >= .05))
print("Mean MEDV by CHAS:")
display(boston_df.groupby("CHAS")["MEDV"].agg(["count", "mean", "median"]).round(2))
print(f"t statistic = {t_stat:.4f}; p-value = {t_p:.6g}")
print("H0: mean MEDV is the same for CHAS=0 and CHAS=1")
print("Decision:", "Reject H0; evidence of a mean difference." if t_p < .05 else "Do not reject H0; difference not significant.")

**Interpretation:** The t-test checks the difference in **mean** MEDV across CHAS groups. Although the business prompt says “median value,” MEDV is the name of the response variable; the ordinary t-test targets its group means.

### Question 14. ANOVA for MEDV across three AGE groups

In [ ]:
age_model = ols("MEDV ~ C(AGE_GROUP)", data=boston_df).fit()
anova_table = sm.stats.anova_lm(age_model, typ=2)
display(anova_table)
anova_p = float(anova_table.loc["C(AGE_GROUP)", "PR(>F)"])
print("H0: population mean MEDV is equal across all three AGE groups")
print("Decision:", "Reject H0; at least one group mean differs." if anova_p < .05 else "Do not reject H0; differences are not significant.")

### Question 15. Pearson correlation of NOX and INDUS

In [ ]:
valid = boston_df[["NOX", "INDUS"]].dropna()
r, pearson_p = stats.pearsonr(valid["NOX"], valid["INDUS"])
print(f"Pearson r = {r:.4f}; p-value = {pearson_p:.6g}")
print("H0: population Pearson correlation is zero")
print("Decision:", "Reject H0; evidence of linear association." if pearson_p < .05 else "Do not reject H0; association not statistically significant.")

### Question 16. Simple linear regression: DIS predicting MEDV

In [ ]:
regression_data = boston_df[["MEDV", "DIS"]].dropna()
regression = ols("MEDV ~ DIS", data=regression_data).fit()
print(regression.summary())
print(f"Slope (DIS) = {regression.params['DIS']:.4f}")
print(f"Slope p-value = {regression.pvalues['DIS']:.6g}")
print(f"R-squared = {regression.rsquared:.4f}")
print("Interpretation: A one-unit increase in DIS is associated with", round(regression.params["DIS"], 3), "units of MEDV (thousand USD), on average, in this unadjusted model.")
plt.figure(figsize=(8, 5))
sns.regplot(data=regression_data, x="DIS", y="MEDV", scatter_kws={"alpha": .5}, line_kws={"color": "darkred"})
plt.title("Simple linear regression: DIS vs. MEDV")
plt.xlabel("Weighted distance to Boston employment centers (DIS)")
plt.ylabel("MEDV (thousands of USD)")
plt.tight_layout()
plt.show()

**Interpretation:** The fitted slope is the expected change in MEDV for one additional DIS unit, not proof that moving a home farther from employment centers changes its price. Other housing and neighborhood characteristics could confound this association.

## Management summary

In [ ]:
print("BOSTON HOUSING — SUMMARY")
print(f"CHAS and MEDV: t-test p = {t_p:.3g}; mean difference (CHAS 1 - 0) = {chas1.mean()-chas0.mean():.2f} thousand USD")
print(f"AGE groups and MEDV: ANOVA p = {anova_p:.3g}")
print(f"NOX and INDUS: Pearson r = {r:.3f}, p = {pearson_p:.3g}")
print(f"DIS and MEDV: linear regression slope = {regression.params['DIS']:.3f}, p = {regression.pvalues['DIS']:.3g}; R² = {regression.rsquared:.3f}")
print("These are historical, unadjusted observational associations. Further analysis is required before making policy or investment decisions.")